# DubFlow — Google Colab AI service

Run all cells. Colab provides Whisper transcription, NLLB or SeamlessM4T
translation, and one of seven speech engines. The final cell prints the three
values required by the local application's `.env`.

The base install covers every Whisper checkpoint, both translation engines, and
the `mms` voice. The other voices are optional — turn them on in the next cell.
`coqui` and `f5` pull large dependency trees and sometimes downgrade Colab's
preinstalled torch; install only the ones you actually plan to pick.

This server can also run the whole pipeline on its own: `POST /jobs` with a video, poll `GET /jobs/{id}`, then fetch `GET /jobs/{id}/download`. Nothing has to run on your machine for that route, but Colab storage is ephemeral - download results before the session ends.

In [ ]:
REPO_URL = "https://github.com/huynhphatloi/MultilingualVideoDubbingSystem.git"
BRANCH = "main"
PORT = 8000

# Fallback only. The form sends a Whisper checkpoint with every request.
WHISPER_MODEL = "small"

# Optional voice engines. 'mms' always works without any of these.
INSTALL_EDGE = True    # edge   - most natural, no GPU, needs internet per call
INSTALL_PIPER = False  # piper  - fast local voices
INSTALL_COQUI = False  # xtts_v2, vixtts - voice cloning
INSTALL_F5 = False     # f5_vi, f5_base  - flow-matching cloning

In [ ]:
from pathlib import Path

if Path("/content/dubflow").exists():
    # Never keep an earlier checkout: both the API contract and the stage code
    # live in this repo, and a stale one fails in ways that look like AI errors.
    !git -C /content/dubflow fetch --depth 1 origin {BRANCH} && git -C /content/dubflow reset --hard FETCH_HEAD
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/dubflow
%cd /content/dubflow/colab
!pip install -q -r requirements.txt

In [ ]:
# Optional engines. Each one is skipped unless its flag is on above.
if INSTALL_EDGE:
    !pip install -q edge-tts
if INSTALL_PIPER:
    !pip install -q piper-tts
if INSTALL_COQUI:
    !pip install -q coqui-tts
if INSTALL_F5:
    !pip install -q f5-tts
print('Optional engines ready. /health lists which ones the server can see.')

In [ ]:
import os
import secrets
import subprocess
import threading
import time

import uvicorn

AUTH_TOKEN = secrets.token_urlsafe(24)
os.environ["AUTH_TOKEN"] = AUTH_TOKEN
os.environ["WHISPER_MODEL"] = WHISPER_MODEL

import server

# `import` is a no-op once the module is cached, so re-running this cell would
# print a fresh token while the server kept checking the one from the first run
# - every call then comes back "401 bad token". Push the current one in.
server.AUTH_TOKEN = AUTH_TOKEN

if "api_thread" not in globals() or not api_thread.is_alive():
    api_thread = threading.Thread(
        target=lambda: uvicorn.run(server.app, host="0.0.0.0", port=PORT, log_level="warning"),
        daemon=True,
    )
    api_thread.start()
    time.sleep(2)
print("Colab AI API started. Models load on the first request.")

In [ ]:
import re

cloudflared = Path("/content/cloudflared")
if not cloudflared.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

if "tunnel" in globals():
    tunnel.terminate()

tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for line in iter(tunnel.stdout.readline, ""):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Cloudflare tunnel did not return a public URL")

# One line to copy: it writes .env, restarts the API, and reports what is live.
print("Run this in the project directory on your machine:\n")
print(f"  make colab URL={public_url} TOKEN={AUTH_TOKEN}\n")
print("Or set these by hand in .env:")
print(f"  COLAB_API_URL={public_url}")
print(f"  COLAB_API_TOKEN={AUTH_TOKEN}")